# GVC Measure 

Pulls OECD Trade-in-Value-Added (TiVa) API from each sector mapping to HS-6 Product Level in order to develop and create a GVC-intensity scoring measure.



In [1]:
# Pull BACII data from the database for 2017


import datetime
import os
import sys
import time 
import numpy as np
import pandas as pd
import sqlalchemy as sa
from sqlalchemy import create_engine
from sqlalchemy import MetaData, Table, Column, Integer, String, DateTime, Float
from sqlalchemy.sql import select, and_, or_, not_
from sqlalchemy import func
from sqlalchemy import text
from sqlalchemy import inspect
from sqlalchemy import exc  
from sqlalchemy import event
from sqlalchemy.pool import Pool
import logging

logging.basicConfig()
logging.getLogger('sqlalchemy.engine').setLevel(logging.INFO)

## Data Loader for Project Codes

In [2]:
import sys
from pathlib import Path

# Works on Mac, Windows, Linux — no hardcoded paths
PROJECT_ROOT = Path.cwd().parent  # adjust depth if notebook is in a subfolder
sys.path.insert(0, str(PROJECT_ROOT))

from config import DATA_DIR, OUTPUT_DIR

# Use like this — path separators handled automatically
baci_file = DATA_DIR / "BACI_HS17_V202601" / "product_codes_HS17_V202601.csv"

In [ ]:
import pandas as pd

def load_product_codes():
    # Load HS92 product codes from data folder
    file_path = 'data/BACI_HS92_V202601/product_codes_HS92_V202601.csv'

    try:
        df = pd.read_csv(file_path, dtype='str')
        # Fix leading zeros
        df['code'] = df['code'].str.zfill(6)
        # Check all codes are exactly 6 digits
        lengths = df['code'].str.len().value_counts()
        print("Code length distribution:")
        print(lengths)
        print(f"Loaded {len(df)} product codes")
        print(df.head(20))
        return df

    except FileNotFoundError:
        print(f"File not found: {file_path}")
        print("Available files:")
        import os
        data_dir = 'data/BACI_HS92_V202601'
        if os.path.exists(data_dir):
            print(os.listdir(data_dir))

df_hs92 = load_product_codes()

In [ ]:
# ── HS92 lookup — used by anchor smoke test ───────────────────────────────────
# Keyed on the same codes as df_layer0, so smoke test uses identical descriptions.
hs92_lookup = df_hs92.set_index("code")["description"].to_dict()

def get_desc(code: str) -> str:
    code = str(code).zfill(6)
    desc = hs92_lookup.get(code)
    if desc is None:
        raise KeyError(f"Code {code} not found in HS92")
    return desc

## HS Chapter, Subheadings, Descriptions

In [ ]:
import requests
import pandas as pd

# ── Layer 0 Chapter Prior ─────────────────────────────────────────────────────
CLASS_NAMES = {
    1: "Raw/Primary",
    2: "Processed Material",
    3: "Component/Part",
    4: "Intermediate Assembly/Capital",
    5: "Final Good",
}

CHAPTER_PRIOR = [
    ( 1, 27, 1),   # live animals, food, raw materials, minerals, fuels
    (28, 40, 2),   # chemicals, plastics, rubber
    (41, 60, 3),   # leather, wood, paper, textiles (fabrics/yarns)
    (61, 63, 5),   # apparel and clothing accessories
    (64, 67, 5),   # footwear, headgear
    (68, 83, 3),   # stone, glass, ceramics, base metals, metal articles
    (84, 92, 4),   # machinery, electrical equipment, instruments
    (93, 93, 4),   # arms and ammunition
    (94, 96, 5),   # furniture, toys, miscellaneous manufactures
    (97, 97, 5),   # art, antiques
]

chapter_prior = pd.DataFrame([
    {"chapter": str(ch).zfill(2), "prior_class": cls, "prior_label": CLASS_NAMES[cls]}
    for lo, hi, cls in CHAPTER_PRIOR
    for ch in range(lo, hi + 1)
])

print("Chapter prior class distribution:")
print(chapter_prior.groupby(["prior_class", "prior_label"])["chapter"]
      .count().rename("n_chapters").to_string())

# ── Source 1: GitHub CSV → chapter and heading text ───────────────────────────
print("\nFetching GitHub HS hierarchy CSV...")
gh = pd.read_csv(
    "https://raw.githubusercontent.com/datasets/harmonized-system/main/data/harmonized-system.csv",
    dtype=str
)
gh["level"] = gh["level"].str.strip()

chapter_text = (gh[gh["level"] == "2"]
    .rename(columns={"hscode": "chapter", "description": "chapter_text"})
    [["chapter", "chapter_text"]])

heading_text = (gh[gh["level"] == "4"]
    .rename(columns={"hscode": "heading", "description": "heading_text"})
    [["heading", "heading_text"]])

print(f"  Chapters: {len(chapter_text)}, Headings: {len(heading_text)}")

# ── Source 2: HS92 directly (no WITS) ────────────────────────────────────────
# Use df_hs92 loaded above — clean single-revision codes, no multi-year noise.
df_sub = (df_hs92
    .rename(columns={"code": "subheading", "description": "subheading_text"})
    .copy())
df_sub["chapter"] = df_sub["subheading"].str[:2]
df_sub["heading"]  = df_sub["subheading"].str[:4]
print(f"  Subheadings (HS92): {len(df_sub):,}")   # should print 5,022

# ── HS92 heading coverage check + OEC patch ───────────────────────────────────
hs92_headings   = df_hs92["code"].str[:4].unique()
github_headings = set(heading_text["heading"].values)
missing_hd      = [h for h in hs92_headings if h not in github_headings]
print(f"\nHS92 headings not in GitHub CSV: {len(missing_hd)}")
print(sorted(missing_hd))

if missing_hd:
    oec_url   = "https://api-v2.oec.world/tesseract/members?cube=trade_i_baci_a_92&level=HS4"
    oec       = pd.DataFrame(requests.get(oec_url, timeout=30).json()["data"])
    id_col    = [c for c in oec.columns if c.endswith(" ID")][0]
    nm_col    = [c for c in oec.columns if not c.endswith(" ID")][0]
    oec_patch = oec.rename(columns={id_col: "raw_id", nm_col: "heading_text"})
    oec_patch["heading"] = oec_patch["raw_id"].astype(str).str[-4:].str.zfill(4)
    oec_patch = oec_patch[oec_patch["heading"].isin(missing_hd)][["heading", "heading_text"]]
    heading_text = pd.concat([heading_text, oec_patch], ignore_index=True).drop_duplicates("heading")
    print(f"Patched {len(oec_patch)} headings from OEC HS92")

# ── Assemble Layer 0 ──────────────────────────────────────────────────────────
df_layer0 = (df_sub
    .merge(chapter_text,  on="chapter", how="left")
    .merge(heading_text,   on="heading",  how="left")
    .merge(chapter_prior,  on="chapter",  how="left")
    [["chapter", "chapter_text",
      "heading",  "heading_text",
      "subheading", "subheading_text",
      "prior_class", "prior_label"]])

# ── Sanity checks ─────────────────────────────────────────────────────────────
print(f"\nRows:                  {len(df_layer0):,}")
print(f"Missing chapter text:  {df_layer0['chapter_text'].isna().sum()}")
print(f"Missing heading text:  {df_layer0['heading_text'].isna().sum()}")
print(f"Unassigned prior:      {df_layer0['prior_class'].isna().sum()}")
print(f"\nPrior class distribution (products):")
print(df_layer0.groupby(["prior_class","prior_label"], dropna=False)["subheading"]
      .count().rename("n_products").to_string())

df_layer0.head(10)

## Layer 1 — Zero-Shot NLI on HS6 Descriptions

Runs each HS6 short description through a zero-shot Natural Language Inference (NLI) model against five chain-position hypotheses (one per class). No training data is required — classification is driven entirely by the semantic relationship between the product description and the explicitly stated hypothesis.

**Output columns added to `df_layer1`:**
- `nli_prob_1` … `nli_prob_5` — normalised entailment probability per class  
- `nli_continuous` — probability-weighted average of class indices [1, 5]  
- `nli_class` — argmax predicted class (1–5)  
- `nli_label` — human-readable class name  
- `nli_uncertainty` — normalised entropy H / log(5), bounded [0, 1]

**Models:** `cross-encoder/nli-deberta-v3-small` (primary, open weights).  
`facebook/bart-large-mnli` is defined as a second model for ensemble comparison in later work.  
Both are run locally with no API dependency.

In [6]:
# ── Layer 1: Hypothesis definitions ──────────────────────────────────────────
# Each hypothesis is the natural-language statement of what it means for a
# product to belong to that chain-position class. Phrasing is directly from
# the Master Plan §4 Layer 1. Tested empirically against anchor products —
# performance is sensitive to wording, so treat these as v1 to be iterated.

HYPOTHESES = {
    # ── Classes 1 and 2: unchanged from the 64% run ──────────────────────────
    1: (
        "This product is a raw or primary commodity in its natural or near-natural state, "
        "obtained by extraction, harvesting, or primary production. It has undergone no "
        "significant industrial transformation — it may be cleaned, dried, sorted, or "
        "concentrated, but its fundamental chemical or physical form is unchanged. "
        "Product descriptions typically use terms such as 'live', 'fresh', 'crude', "
        "'raw', 'unrefined', 'ore', or 'unprocessed'. "
        "Examples: live animals, crude petroleum, iron ore, raw cotton, uncut timber, "
        "fresh or chilled fish, unroasted coffee beans, coal, natural gas."
    ),
    2: (
        "This product is a processed or refined industrial material — a homogeneous bulk "
        "input that has been chemically or physically transformed from a raw material into "
        "a standardised form suitable for further manufacturing. It is traded by weight or "
        "volume rather than as individual functional units, and it is not yet shaped into "
        "a specific part or component. Product descriptions typically use terms such as "
        "'refined', 'processed', 'wrought', 'unwrought', 'rolled', 'drawn', 'alloy', "
        "'compound', 'preparation', 'yarn', or 'fabric'. "
        "Examples: refined petroleum, cold-rolled steel sheet, polyethylene resin, "
        "cotton yarn, aluminium ingots, sulphuric acid, woven fabric, rubber compound."
    ),

    # ── Class 3: add B2B qualifier without removing 'parts' language ──────────
    # Fix: bearings and PCBs scored Class 5 because they're finished products.
    # Added: "purchased by manufacturers not end consumers" as secondary qualifier.
    # Kept: 'parts and accessories' language — it is still a strong signal.
    3: (
        "This product is a discrete manufactured part, component, or accessory specifically "
        "designed to be incorporated into a larger machine, vehicle, or system during "
        "assembly. It may be a finished product in its own right but is purchased by "
        "manufacturers rather than end consumers. Product descriptions frequently use "
        "terms such as 'parts', 'parts and accessories', 'components', 'fittings', or "
        "'for use in', or name a specific mechanical or electronic element. "
        "Examples: ball bearings, printed circuit boards, hydraulic valves, pistons, "
        "optical lenses, gaskets, semiconductor wafers, fasteners, gears, engine parts."
    ),

    # ── Class 4: add machine-tool language explicitly ─────────────────────────
    # Fix: 'machine-tools for working metal' scored Class 2 because 'material'
    # triggered the processed material hypothesis.
    # Added: 'machine-tools for working', 'apparatus for' as explicit signals.
    4: (
        "This product is a complete functional machine, engine, apparatus, or machine-tool "
        "— such as a diesel engine, CNC machine-tool, industrial robot, compressor, or "
        "printing press — purchased by producers to manufacture goods, work materials, or "
        "deliver services. Descriptions often reference an industrial process: "
        "'machines for', 'apparatus for', 'machine-tools for working'. "
        "It functions independently but is used as capital equipment in production, "
        "not sold to final consumers."
    ),

    # ── Class 5: keep direct-use framing, add telephone explicitly ───────────
    # Fix: 'telephone sets for cellular networks' scored Class 2 — 'networks'
    # read as B2B infrastructure. Added 'telephone set, mobile phone' explicitly.
    5: (
        "This product is a finished good at the end of the production chain — such as a "
        "passenger car, mobile phone, telephone set, washing machine, television, clothing, "
        "bottled wine, or packaged food — purchased for direct consumption or use without "
        "any further industrial transformation or incorporation into another product. "
        "The buyer uses it directly, not as an input to manufacture something else."
    ),
}

CLASS_NAMES = {
    1: "Raw/Primary",
    2: "Processed Material",
    3: "Component/Part",
    4: "Intermediate Assembly/Capital",
    5: "Final Good",
}

print("Hypotheses defined:")
for cls, hyp in HYPOTHESES.items():
    print(f"  Class {cls} ({CLASS_NAMES[cls]}): {hyp[:60]}...")

Hypotheses defined:
  Class 1 (Raw/Primary): This product is a raw or primary commodity in its natural or...
  Class 2 (Processed Material): This product is a processed or refined industrial material —...
  Class 3 (Component/Part): This product is a discrete manufactured part, component, or ...
  Class 4 (Intermediate Assembly/Capital): This product is a complete functional machine, engine, appar...
  Class 5 (Final Good): This product is a finished good at the end of the production...


In [7]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "transformers", "sentencepiece"],
    capture_output=True, text=True
)
print(result.stdout[-3000:])
print(result.stderr[-3000:])

 ./.venv/lib/python3.13/site-packages (from transformers) (2026.4.4)


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip



In [9]:
## ── Layer 1: Load NLI model ───────────────────────────────────────────────────
# Primary model: DeBERTa-v3-small fine-tuned on MNLI + SNLI + ANLI.
# Chosen for: strong zero-shot NLI accuracy, small footprint (~180MB),
# locally runnable, open weights (Apache 2.0).
# Fallback / ensemble comparison model: facebook/bart-large-mnli.
# Not loaded by default — uncomment to run ensemble (slower, ~1.6GB).
# multi_label=True: each hypothesis is scored independently rather than
# as competing candidates. This is correct here because a description can
# plausibly entail more than one hypothesis at partial probability —
# we normalise the raw scores to a probability distribution ourselves.

#  Run this in its own cell before loading any model
# device.py  (project root, committed to GitHub)
import torch
from transformers import pipeline

def get_device() -> str:
    if torch.cuda.is_available(): return "cuda"
    if torch.backends.mps.is_available(): return "mps"
    return "cpu"

DEVICE = get_device()
print(f"Device: {DEVICE}")

if "nli" not in dir() or nli is None:
    model_id = "MoritzLaurer/deberta-v3-base-zeroshot-v1"
    print(f"Loading {model_id}...")
    nli = pipeline(
        "zero-shot-classification",
        model=model_id,
        device=DEVICE,
    )
    print("Model ready.")

# ── Updated scoring function ──────────────────────────────────────────────────
def nli_score_product(description: str) -> dict:
    result = nli(
        description,
        candidate_labels=list(HYPOTHESES.values()),
        multi_label=False,    # softmax — forces single class selection
    )
    # Map scores back to class numbers
    hyp_to_class = {v: k for k, v in HYPOTHESES.items()}
    scores = {
        hyp_to_class[label]: score
        for label, score in zip(result["labels"], result["scores"])
    }
    # Continuous score: probability-weighted average
    continuous = sum(cls * p for cls, p in scores.items())
    # Uncertainty: entropy of distribution
    import math
    entropy = -sum(p * math.log(p + 1e-9) for p in scores.values())
    uncertainty = entropy / math.log(len(scores))   # normalise to [0,1]

    predicted_class = max(scores, key=scores.get)
    return {
        "nli_class":      predicted_class,
        "nli_label":      CLASS_NAMES[predicted_class],
        "nli_continuous": round(continuous, 3),
        "nli_uncertainty": round(uncertainty, 3),
        "class_probs":    {cls: round(p, 4) for cls, p in sorted(scores.items())},
    }

/Users/apueelawekulom/Github/masters_thesis/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps
Loading MoritzLaurer/deberta-v3-base-zeroshot-v1...


Loading weights: 100%|██████████| 202/202 [00:00<00:00, 4812.03it/s]


Model ready.


In [10]:
# ── Layer 1: Scoring function ─────────────────────────────────────────────────
import math

def nli_score_product(description: str) -> dict:
    """
    Score one HS6 product description against all five chain-position hypotheses.

    Returns a dict with:
      nli_prob_1..5   normalised entailment probability per class
      nli_continuous  probability-weighted average of class indices [1, 5]
      nli_class       predicted class (int 1-5, argmax)
      nli_label       human-readable class name
      nli_uncertainty normalised entropy H / log(5), bounded [0, 1]

    Returns all-None dict for empty or null descriptions.
    """
    desc = str(description).strip() if description and str(description).strip() else None
    if desc is None:
        return {
            "nli_prob_1": None, "nli_prob_2": None, "nli_prob_3": None,
            "nli_prob_4": None, "nli_prob_5": None,
            "nli_continuous":  None,
            "nli_class":       None,
            "nli_label":       None,
            "nli_uncertainty": None,
        }

    # multi_label=True: independent entailment score per hypothesis
    result = nli(
        desc,
        candidate_labels=list(HYPOTHESES.values()),
        multi_label=True,
    )

    # Map hypothesis text back to class number
    hyp_to_class = {v: k for k, v in HYPOTHESES.items()}
    raw = {hyp_to_class[label]: score
           for label, score in zip(result["labels"], result["scores"])}

    # Normalise raw entailment scores to sum to 1
    total = sum(raw.values())
    norm  = {cls: raw[cls] / total for cls in sorted(raw)}

    # Continuous score: probability-weighted average of class indices
    continuous = sum(cls * norm[cls] for cls in norm)

    # Predicted class: argmax of normalised probabilities
    predicted = max(norm, key=norm.get)

    # Uncertainty: normalised Shannon entropy H / log(5), bounded [0, 1]
    # High uncertainty (→ 1.0) = model spreads probability evenly across classes
    # Low uncertainty (→ 0.0)  = model concentrates on one class
    entropy     = -sum(p * math.log(p + 1e-12) for p in norm.values())
    max_entropy = math.log(len(norm))
    uncertainty = entropy / max_entropy if max_entropy > 0 else 0.0

    return {
        "nli_prob_1":      round(norm[1], 6),
        "nli_prob_2":      round(norm[2], 6),
        "nli_prob_3":      round(norm[3], 6),
        "nli_prob_4":      round(norm[4], 6),
        "nli_prob_5":      round(norm[5], 6),
        "nli_continuous":  round(continuous, 4),
        "nli_class":       predicted,
        "nli_label":       CLASS_NAMES[predicted],
        "nli_uncertainty": round(uncertainty, 4),
    }

print("nli_score_product() defined.")

nli_score_product() defined.


In [ ]:
# ── Layer 1: Anchor smoke test (expanded, HS92-verified) ─────────────────────
# Run BEFORE full scoring to validate hypotheses.
# Must reach ≥80% before proceeding to full HS92 run.
# ANCHOR_CODES is defined here and reused in the formal validation cell below.

ANCHOR_CODES = [
    # (hs6_code, expected_class, note)

    # Class 1 — Raw / Primary
    ("270900", 1, "canonical — crude petroleum"),
    ("260111", 1, "canonical — iron ore"),
    ("520100", 1, "canonical — raw cotton"),
    ("010121", 1, "live animal"),
    ("100590", 1, "agricultural commodity — maize"),
    ("240110", 1, "tobacco, unmanufactured"),
    ("440320", 1, "wood in the rough"),
    ("260300", 1, "copper ore, pre-smelting"),

    # Class 2 — Processed Material
    ("740811", 2, "canonical — refined copper wire"),
    ("720811", 2, "canonical — hot-rolled steel sheet"),
    ("390110", 2, "polyethylene resin"),
    ("720110", 2, "pig iron"),
    ("480411", 2, "kraft paper"),
    ("520511", 2, "cotton yarn"),
    ("281000", 2, "phosphorus oxides — industrial chemical"),

    # Class 3 — Component / Part
    ("848210", 3, "canonical — ball bearings"),
    ("854231", 3, "integrated circuits"),
    ("848180", 3, "valves"),
    ("401110", 3, "pneumatic tyres"),
    ("854430", 3, "insulated wire"),
    ("870899", 3, "parts of motor vehicles"),
    ("840991", 3, "parts for spark-ignition engines"),
    ("900190", 3, "optical lenses"),

    # Class 4 — Intermediate Assembly / Capital
    ("840820", 4, "canonical — compression-ignition engine"),
    ("847710", 4, "injection moulding machines"),
    ("842952", 4, "excavators"),
    ("844110", 4, "printing machinery"),
    ("841111", 4, "turbojets"),
    ("851521", 4, "welding machines"),
    ("903180", 4, "measuring instruments"),

    # Class 5 — Final Good
    ("870322", 5, "canonical — passenger cars"),
    ("851120", 5, "canonical — telephone apparatus (HS92 phones)"),
    ("220421", 5, "canonical — bottled wine"),
    ("640399", 5, "footwear"),
    ("610910", 5, "T-shirts"),
    ("950300", 5, "toys"),
    ("220300", 5, "beer"),
    ("940360", 5, "furniture"),
]

# Build ANCHORS from WITS lookup (same descriptions used in full scoring)
ANCHORS, missing_codes = [], []
for code, expected_class, note in ANCHOR_CODES:
    try:
        ANCHORS.append((get_desc(code), expected_class, code, note))
    except KeyError:
        missing_codes.append(code)

if missing_codes:
    print(f"WARNING — not in HS92: {missing_codes}\n")

print(f"Running anchor smoke test on {len(ANCHORS)} products...")
print(f"{'Description':<55} {'Exp':>4} {'Got':>4} {'Score':>7} {'Unc':>6}  OK?")
print("-" * 90)

results = []
for desc, expected, code, note in ANCHORS:
    r       = nli_score_product(desc)
    got     = r["nli_class"]
    correct = got == expected
    print(f"{desc[:54]:<55} {expected:>4} {got:>4} "
          f"{r['nli_continuous']:>7.3f} {r['nli_uncertainty']:>6.3f}  "
          f"{'✓' if correct else '✗'}"
          f"{' [' + note + ']' if not correct else ''}")
    results.append({"code": code, "description": desc,
                    "expected": expected, "got": got,
                    "score": r["nli_continuous"],
                    "uncertainty": r["nli_uncertainty"],
                    "correct": correct, "note": note})

df_smoke = pd.DataFrame(results)
accuracy  = df_smoke["correct"].mean()
print(f"\nAnchor accuracy: {df_smoke['correct'].sum()}/{len(df_smoke)} = {accuracy:.0%}")
print("\nPer-class accuracy:")
print(df_smoke.groupby("expected").apply(
    lambda x: f"  Class {x.name}: {x['correct'].sum()}/{len(x)} = {x['correct'].mean():.0%}"
).to_string(header=False))
print("\nProceed to full scoring only if accuracy ≥ 80%.")

In [ ]:
# ── Layer 1: Score full HS6 universe ─────────────────────────────────────────
# Applies nli_score_product() to every row of df_layer0 using the
# subheading_text column (HS6 short description from WITS).
#
# Runtime note: ~5,000 products × ~0.3s per product ≈ 25 min on CPU.
# Set device=0 in the pipeline call above if a GPU is available (~2–3 min).
# A tqdm progress bar is shown so you can monitor progress.
#
# The scored DataFrame is saved to CSV immediately after completion so
# results are not lost if the kernel is restarted.

import pandas as pd
from tqdm.auto import tqdm

tqdm.pandas(desc="NLI scoring")

print(f"Scoring {len(df_layer0):,} products ...")

# Score each row and expand the result dict into columns
nli_results = df_layer0["subheading_text"].progress_apply(nli_score_product)
df_nli      = pd.DataFrame(nli_results.tolist())

# Concatenate with Layer 0 — preserve all existing columns, append NLI columns
df_layer1 = pd.concat([df_layer0.reset_index(drop=True),
                        df_nli.reset_index(drop=True)], axis=1)

print(f"Layer 1 shape: {df_layer1.shape}")
print(f"Missing NLI scores: {df_layer1['nli_class'].isna().sum()}")
print(f"Predicted class distribution:")
print(
    df_layer1
    .groupby(["nli_class", "nli_label"], dropna=False)["subheading"]
    .count()
    .rename("n_products")
    .to_string()
)

# Save immediately
out_path = "layer1_nli_scores.csv"
df_layer1.to_csv(out_path, index=False)
print(f"Saved to {out_path}")

In [ ]:
# ── Layer 1: Score full HS92 corpus ───────────────────────────────────────────
# Applies nli_score_product() to every row of df_layer0.
# Runtime: ~5,022 products × ~0.3s ≈ 25 min CPU / ~3–5 min MPS.
# Saves immediately on completion to avoid losing results on kernel restart.

import pandas as pd
from tqdm.auto import tqdm

tqdm.pandas(desc="NLI scoring")

print(f"Scoring {len(df_layer0):,} products ...")

nli_results = df_layer0["subheading_text"].progress_apply(nli_score_product)
df_nli      = pd.DataFrame(nli_results.tolist())

df_layer1 = pd.concat([df_layer0.reset_index(drop=True),
                        df_nli.reset_index(drop=True)], axis=1)

print(f"Layer 1 shape: {df_layer1.shape}")
print(f"Missing NLI scores: {df_layer1['nli_class'].isna().sum()}")
print(f"Predicted class distribution:")
print(
    df_layer1
    .groupby(["nli_class", "nli_label"], dropna=False)["subheading"]
    .count()
    .rename("n_products")
    .to_string()
)

out_path = "layer1_nli_scores_hs92.csv"
df_layer1.to_csv(out_path, index=False)
print(f"Saved to {out_path}")

In [ ]:
# ── Layer 1: Diagnostics ──────────────────────────────────────────────────────
# Examines agreement between Layer 0 chapter prior and Layer 1 NLI predictions,
# and flags high-uncertainty products for inspection.

import pandas as pd

# 1. Agreement rate between chapter prior and NLI prediction
df_layer1["prior_nli_agree"] = (
    df_layer1["prior_class"] == df_layer1["nli_class"]
)
agree_rate = df_layer1["prior_nli_agree"].mean()
print(f"Chapter prior / NLI agreement rate: {agree_rate:.1%}")
print("(Expected to be moderate — NLI adds within-chapter resolution)")

# 2. Disagreement breakdown by chapter
disagree = df_layer1[~df_layer1["prior_nli_agree"]].copy()
print(f"Disagreements: {len(disagree):,} products ({len(disagree)/len(df_layer1):.1%})")
print("Top 10 chapters by disagreement count:")
print(
    disagree.groupby("chapter")["subheading"]
    .count()
    .sort_values(ascending=False)
    .head(10)
    .to_string()
)

# 3. High-uncertainty products (uncertainty > 0.75 = model very unsure)
HIGH_UNC_THRESHOLD = 0.75
high_unc = df_layer1[df_layer1["nli_uncertainty"] > HIGH_UNC_THRESHOLD].copy()
print(f"High-uncertainty products (unc > {HIGH_UNC_THRESHOLD}): {len(high_unc):,}")
print("Sample of high-uncertainty products (likely double-dipping candidates):")
print(
    high_unc[["subheading", "subheading_text", "nli_class",
               "nli_label", "nli_uncertainty"]]
    .sort_values("nli_uncertainty", ascending=False)
    .head(15)
    .to_string(index=False)
)

# 4. Continuous score distribution by predicted class
print("Continuous score statistics by NLI class:")
print(
    df_layer1
    .groupby("nli_class")["nli_continuous"]
    .agg(["mean", "std", "min", "max"])
    .round(3)
    .to_string()
)

df_layer1.head(10)

In [ ]:
# ── Layer 1: Formal anchor validation against scored dataset ──────────────────
# Looks up ANCHOR_CODES in df_layer1 by subheading code after full scoring.
# Uses the same ANCHOR_CODES defined in the smoke test cell above.

scored = df_layer1.set_index("subheading")

print(f"{'Code':<8} {'Exp':>4} {'Got':>4} {'Score':>7} {'Unc':>6}  OK?  Description")
print("-" * 95)

n_correct, n_found = 0, 0
for code, expected, note in ANCHOR_CODES:
    code = str(code).zfill(6)
    if code not in scored.index:
        print(f"{code:<8}  not found in scored dataset")
        continue
    row     = scored.loc[code]
    got     = int(row["nli_class"])
    correct = got == expected
    if correct: n_correct += 1
    n_found += 1
    desc = str(row["subheading_text"])[:40]
    print(f"{code:<8} {expected:>4} {got:>4} "
          f"{row['nli_continuous']:>7.3f} {row['nli_uncertainty']:>6.3f}  "
          f"{'✓' if correct else '✗'}   {desc}")

print(f"\nFormal anchor accuracy: {n_correct}/{n_found} = "
      f"{n_correct/n_found:.0%}" if n_found else "No anchors found.")
print("Target: ≥80% before proceeding to Layer 2.")